# broadcast-initial-weights — ex2: broadcast a full state_dict — params AND BN buffers

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `broadcast-initial-weights`. Running the final beacon cell reports progress against the `Distributed: broadcast initial weights` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: broadcast initial weights` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`broadcast-initial-weights`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "broadcast-initial-weights"
DD_SUBTOPIC = "Distributed: broadcast initial weights"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Broadcasting a full `state_dict` — params + buffers

Ex1 broadcast each `nn.Parameter.data`. That misses two things in a real DDP init:

1. **BatchNorm running stats** (`running_mean`, `running_var`, `num_batches_tracked`) are NOT parameters — they live in `model.buffers()`. Ranks must agree on them too, or the first eval batch sees rank-specific BN stats.
2. **Non-float buffers** (LongTensors, mask buffers, etc.) — must still be in sync.

Canonical pattern: iterate `model.state_dict().values()` (which yields params + persistent buffers) and broadcast each tensor:

```python
for tensor in model.state_dict().values():
    dist.broadcast(tensor, src=0)
```

**Why state_dict, not parameters + buffers.** `state_dict()` returns the canonical superset in a deterministic order — same order on every rank because every rank has the same model graph. Zipping per-rank lists from `parameters()` and `buffers()` separately is more typing for the same answer.

**state_dict tensors are VIEWS into the underlying storage.** Mutating them in-place (which is what `dist.broadcast` does) updates the live model. No `model.load_state_dict()` call needed afterward.

### Exercise 2 — broadcast a full state_dict — params AND BN buffers

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply `dist.broadcast(tensor, src=0)` over every tensor in `model.state_dict().values()` so every rank's parameters AND BatchNorm buffers (`running_mean`, `running_var`) all mirror rank 0 at the start of training.
> Keywords: broadcast, state_dict, BN-buffers, running_mean, DDP-init
> ```

**KCs targeted:** `broadcast-state-dict-values`, `buffers-need-sync-too`

Implement `ex2_broadcast_state_dict(rank, world_size, dist_module, model)`. The full-state DDP init pattern:

1. Iterate `model.state_dict().values()`. Order is deterministic across ranks because every rank has the same model graph.
2. For each yielded tensor, call `dist_module.broadcast(tensor, src=0)`. This mutates the underlying storage in-place; no `load_state_dict` call needed.
3. Return the count of tensors broadcast (used by tests to verify both params AND buffers were visited).

Signature note: `dist_module` is injected (a mocked `dist` on CPU). All calls go via `dist_module.broadcast(...)`, not the global `dist.broadcast`.

Input: `rank`, `world_size` — ints; `dist_module` — torch.distributed or mock; `model` — `nn.Module` with rank-specific initial params AND buffers.
Output: `int` — count of tensors broadcast (i.e. `len(model.state_dict())`).

In [ ]:
def ex2_broadcast_state_dict(rank: int, world_size: int, dist_module, model: 'nn.Module') -> int:
    count = 0
    for tensor in model.state_dict().values():
        dist_module.broadcast(tensor, src=0)
        count += 1
    return count


<details><summary>Solution</summary>

```python
def ex2_broadcast_state_dict(rank: int, world_size: int, dist_module, model: 'nn.Module') -> int:
    count = 0
    for tensor in model.state_dict().values():
        dist_module.broadcast(tensor, src=0)
        count += 1
    return count
```

**Why iterate `state_dict().values()` not `parameters()`.** `parameters()` yields ONLY learnable tensors — Linear weight/bias, Conv weight/bias. It SKIPS BatchNorm running stats, which are persistent buffers, not parameters. Broadcasting params alone leaves a silent bug: per-rank `running_mean` drifts forever.

**`num_batches_tracked` is an `int64` scalar.** Some older backends choke on int broadcast (gloo handles it fine, nccl is finicky). The fake harness above handles all dtypes uniformly via `copy_`.

**Order-determinism.** `state_dict()` returns an `OrderedDict` whose key order is the in-graph traversal order. Every rank has the same model graph, so every rank iterates the same order — no off-by-one or interleaved broadcasts.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()